## `Practice 9
`

In [1]:

!apt-get update -y
!apt-get install -y openmpi-bin libopenmpi-dev

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,328 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:13 http://security.ubuntu.com/ub

In [2]:
%%writefile program.cpp
#include <mpi.h>
#include <iostream>
#include <vector>
#include <random>
#include <cmath>

using namespace std;

int main(int argc, char** argv) {
    // Инициализация MPI
    MPI_Init(&argc, &argv);

    // Ранг текущего процесса и общее количество процессов
    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    // Размер массива случайных чисел
    const long long N = 1'000'000;

    // Начало измерения времени выполнения
    double start_time = MPI_Wtime();

    // Вектор для хранения исходных данных (только на rank 0)
    vector<double> data;

    if (rank == 0) {
        // Генерация массива случайных чисел на процессе с rank = 0
        data.resize(N);

        // Генератор случайных чисел с фиксированным seed
        mt19937 gen(2026);
        uniform_real_distribution<double> dist(0.0, 1.0);

        for (long long i = 0; i < N; i++) {
            data[i] = dist(gen);
        }
    }

    // Векторы для описания распределения данных между процессами
    // counts[i] — количество элементов, передаваемых i-му процессу
    // displs[i] — смещение (начальный индекс) для i-го процесса
    vector<int> counts(size);
    vector<int> displs(size);

    // Базовое количество элементов на процесс
    long long base = N / size;
    // Остаток, если N не делится нацело
    int rem = N % size;

    int offset = 0;
    for (int i = 0; i < size; i++) {
        // Первые rem процессов получают на один элемент больше
        counts[i] = base + (i < rem ? 1 : 0);
        displs[i] = offset;
        offset += counts[i];
    }

    // Локальный массив для каждого процесса
    vector<double> local(counts[rank]);

    // Распределение частей массива между процессами
    // Используется MPI_Scatterv для учёта неравномерного распределения
    MPI_Scatterv(
        rank == 0 ? data.data() : nullptr, // исходный массив (только у rank 0)
        counts.data(),                     // количество элементов для каждого процесса
        displs.data(),                     // смещения
        MPI_DOUBLE,                        // тип данных
        local.data(),                      // локальный буфер
        counts[rank],                      // количество элементов для текущего процесса
        MPI_DOUBLE,
        0,                                 // корневой процесс
        MPI_COMM_WORLD
    );

    // Локальные вычисления:
    // сумма элементов и сумма квадратов элементов
    double local_sum = 0.0;
    double local_sumsq = 0.0;

    for (double x : local) {
        local_sum += x;
        local_sumsq += x * x;
    }

    // Глобальные суммы (только на rank 0)
    double global_sum = 0.0;
    double global_sumsq = 0.0;

    // Сбор локальных сумм на процессе rank = 0
    MPI_Reduce(&local_sum, &global_sum, 1, MPI_DOUBLE, MPI_SUM, 0, MPI_COMM_WORLD);
    MPI_Reduce(&local_sumsq, &global_sumsq, 1, MPI_DOUBLE, MPI_SUM, 0, MPI_COMM_WORLD);

    // Конец измерения времени
    double end_time = MPI_Wtime();

    if (rank == 0) {
        // Вычисление среднего значения
        double mean = global_sum / N;

        // Вычисление дисперсии и стандартного отклонения
        double variance = global_sumsq / N - mean * mean;

        // Защита от отрицательного значения из-за ошибок округления
        variance = max(0.0, variance);

        double stddev = sqrt(variance);

        // Вывод результатов
        cout << "N = " << N << ", processes = " << size << endl;
        cout << "Mean = " << mean << endl;
        cout << "Std dev = " << stddev << endl;
        cout << "Execution time = "
             << (end_time - start_time)
             << " seconds" << endl;
    }

    // Завершение работы MPI
    MPI_Finalize();
    return 0;
}


Writing program.cpp


In [3]:
!mpic++ program.cpp -O2 -o program

In [4]:

!mpirun --allow-run-as-root  --oversubscribe -np 2 ./program
!mpirun --allow-run-as-root  --oversubscribe -np 4 ./program
!mpirun --allow-run-as-root  --oversubscribe -np 8 ./program

N = 1000000, processes = 2
Mean = 0.499769
Std dev = 0.288837
Execution time = 0.0376855 seconds
N = 1000000, processes = 4
Mean = 0.499769
Std dev = 0.288837
Execution time = 0.106028 seconds
N = 1000000, processes = 8
Mean = 0.499769
Std dev = 0.288837
Execution time = 0.13245 seconds


## Задание 2

In [2]:
!apt-get update -y
!apt-get install -y openmpi-bin libopenmpi-dev


Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,328 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main 

In [1]:
%%writefile gauss_mpi.cpp
#include <mpi.h>
#include <iostream>
#include <vector>
#include <cmath>

using namespace std;

int main(int argc, char** argv) {
    // Инициализация MPI
    MPI_Init(&argc, &argv);

    // Получаем ранг текущего процесса и общее число процессов
    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    // Размер системы линейных уравнений (NxN)
    const int N = 4;

    // Начало измерения времени выполнения
    double start_time = MPI_Wtime();

    // Матрица коэффициентов A, вектор правых частей b и вектор решения x
    vector<double> A;
    vector<double> b;
    vector<double> x(N);

    // Количество строк матрицы, обрабатываемых одним процессом
    int rows_per_proc = N / size;

    // Локальные части матрицы и вектора для каждого процесса
    vector<double> local_A(rows_per_proc * N);
    vector<double> local_b(rows_per_proc);

    if (rank == 0) {
        // Инициализация системы линейных уравнений на процессе rank = 0
        A = {
            2,  1, -1,   2,
           -3, -1,  2, -11,
           -2,  1,  2,  -3,
            1,  2,  3,   1
        };
        b = {8, -15, -3, 4};
    }

    // Распределение строк матрицы A между процессами
    MPI_Scatter(
        A.data(),                     // исходная матрица (rank = 0)
        rows_per_proc * N,            // количество элементов на процесс
        MPI_DOUBLE,
        local_A.data(),               // локальный буфер
        rows_per_proc * N,
        MPI_DOUBLE,
        0,
        MPI_COMM_WORLD
    );

    // Распределение элементов вектора b между процессами
    MPI_Scatter(
        b.data(),                     // исходный вектор (rank = 0)
        rows_per_proc,
        MPI_DOUBLE,
        local_b.data(),               // локальный буфер
        rows_per_proc,
        MPI_DOUBLE,
        0,
        MPI_COMM_WORLD
    );

    // ========================
    // Прямой ход метода Гаусса
    // ========================
    for (int k = 0; k < N; k++) {
        // Определяем процесс, которому принадлежит ведущая строка
        int owner = k / rows_per_proc;

        // Буфер для хранения ведущей строки и соответствующего элемента b
        vector<double> pivot_row(N + 1);

        if (rank == owner) {
            // Локальный индекс ведущей строки
            int local_k = k % rows_per_proc;

            // Копируем ведущую строку матрицы
            for (int j = 0; j < N; j++)
                pivot_row[j] = local_A[local_k * N + j];

            // Копируем соответствующий элемент вектора b
            pivot_row[N] = local_b[local_k];
        }

        // Передаём ведущую строку всем процессам
        MPI_Bcast(pivot_row.data(), N + 1, MPI_DOUBLE, owner, MPI_COMM_WORLD);

        // Обнуляем элементы ниже главной диагонали
        for (int i = 0; i < rows_per_proc; i++) {
            int global_i = rank * rows_per_proc + i;

            if (global_i > k) {
                double factor = local_A[i * N + k] / pivot_row[k];

                for (int j = k; j < N; j++)
                    local_A[i * N + j] -= factor * pivot_row[j];

                local_b[i] -= factor * pivot_row[N];
            }
        }
    }

    // Сбор преобразованной матрицы и вектора b на процессе rank = 0
    MPI_Gather(
        local_A.data(),
        rows_per_proc * N,
        MPI_DOUBLE,
        A.data(),
        rows_per_proc * N,
        MPI_DOUBLE,
        0,
        MPI_COMM_WORLD
    );

    MPI_Gather(
        local_b.data(),
        rows_per_proc,
        MPI_DOUBLE,
        b.data(),
        rows_per_proc,
        MPI_DOUBLE,
        0,
        MPI_COMM_WORLD
    );

    // ========================
    // Обратный ход метода Гаусса
    // ========================
    if (rank == 0) {
        for (int i = N - 1; i >= 0; i--) {
            x[i] = b[i];

            for (int j = i + 1; j < N; j++)
                x[i] -= A[i * N + j] * x[j];

            x[i] /= A[i * N + i];
        }

        // Конец измерения времени
        double end_time = MPI_Wtime();

        // Вывод решения системы
        cout << "Solution:\n";
        for (int i = 0; i < N; i++)
            cout << "x[" << i << "] = " << x[i] << endl;

        cout << "Execution time: "
             << end_time - start_time
             << " seconds\n";
    }

    // Завершение работы MPI
    MPI_Finalize();
    return 0;
}


Writing gauss_mpi.cpp


In [3]:
!mpic++ gauss_mpi.cpp -o gauss_mpi


In [4]:
!mpirun --allow-run-as-root --oversubscribe -np 2 ./gauss_mpi


Solution:
x[0] = 1
x[1] = 3.5
x[2] = -1.5
x[3] = 0.5
Execution time: 8.2397e-05 seconds


In [5]:
!mpirun --allow-run-as-root --oversubscribe -np 4 ./gauss_mpi


Solution:
x[0] = 1
x[1] = 3.5
x[2] = -1.5
x[3] = 0.5
Execution time: 0.00035585 seconds


## Задание 3

In [6]:
!apt-get update -y
!apt-get install -y openmpi-bin libopenmpi-dev


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,446 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

In [7]:
%%writefile floyd_mpi.cpp
#include <mpi.h>
#include <iostream>
#include <vector>
#include <limits>

using namespace std;

// Константа для "бесконечности" (непрямого пути)
const double INF = 1e9;

int main(int argc, char** argv) {
    MPI_Init(&argc, &argv); // Инициализация MPI

    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank); // Получаем ранг текущего процесса
    MPI_Comm_size(MPI_COMM_WORLD, &size); // Получаем общее количество процессов

    const int N = 4; // Размер графа (NxN)

    double start_time = MPI_Wtime(); // Начало измерения времени

    int rows_per_proc = N / size; // Количество строк, обрабатываемых одним процессом

    vector<double> graph;             // Полная матрица графа (только на rank 0)
    vector<double> local(rows_per_proc * N); // Локальные строки для текущего процесса
    vector<double> full(N * N);       // Полная матрица, собранная на всех процессах

    if (rank == 0) {
        // Инициализация исходного графа на процессе rank = 0
        graph = {
            0,   3,  INF, 7,
            8,   0,  2,   INF,
            5,   INF, 0,  1,
            2,   INF, INF, 0
        };
    }

    // Распределяем строки матрицы между процессами
    MPI_Scatter(graph.data(), rows_per_proc * N, MPI_DOUBLE,
                local.data(), rows_per_proc * N, MPI_DOUBLE,
                0, MPI_COMM_WORLD); // rank 0 отправляет, остальные принимают

    // Инициализация полной матрицы для синхронизации
    MPI_Allgather(local.data(), rows_per_proc * N, MPI_DOUBLE,
                  full.data(), rows_per_proc * N, MPI_DOUBLE,
                  MPI_COMM_WORLD); // все процессы собирают свои локальные строки

    // ================================
    // Основной цикл алгоритма Флойда–Уоршелла
    // ================================
    for (int k = 0; k < N; k++) { // для каждой вершины k
        for (int i = 0; i < rows_per_proc; i++) { // для каждой локальной строки i
            int global_i = rank * rows_per_proc + i; // глобальный индекс строки
            for (int j = 0; j < N; j++) { // для каждой колонки j
                double via_k = full[global_i * N + k] + full[k * N + j]; // расстояние через вершину k
                if (via_k < local[i * N + j]) { // если путь через k короче
                    local[i * N + j] = via_k; // обновляем локальное значение
                }
            }
        }

        // Обмен локальными результатами между всеми процессами
        MPI_Allgather(local.data(), rows_per_proc * N, MPI_DOUBLE,
                      full.data(), rows_per_proc * N, MPI_DOUBLE,
                      MPI_COMM_WORLD); // синхронизация полной матрицы
    }

    double end_time = MPI_Wtime(); // Конец измерения времени

    // Вывод результата (только на rank 0)
    if (rank == 0) {
        cout << "Shortest paths matrix:\n";
        for (int i = 0; i < N; i++) {
            for (int j = 0; j < N; j++) {
                if (full[i * N + j] >= INF / 2) // если путь слишком большой → INF
                    cout << "INF ";
                else
                    cout << full[i * N + j] << " "; // иначе выводим длину пути
            }
            cout << endl;
        }

        cout << "Execution time: "
             << end_time - start_time
             << " seconds\n"; // вывод времени выполнения
    }

    MPI_Finalize(); // Завершение работы MPI
    return 0;
}


Writing floyd_mpi.cpp


In [8]:
!mpic++ floyd_mpi.cpp -o floyd_mpi


In [9]:
!mpirun --allow-run-as-root --oversubscribe -np 2 ./floyd_mpi


Shortest paths matrix:
0 3 5 6 
5 0 2 3 
3 6 0 1 
2 5 7 0 
Execution time: 5.0039e-05 seconds


In [10]:
!mpirun --allow-run-as-root --oversubscribe -np 4 ./floyd_mpi


Shortest paths matrix:
0 3 5 6 
5 0 2 3 
3 6 0 1 
2 5 7 0 
Execution time: 0.00369362 seconds
